# Middleware（中间件）简单来说就是Agent执行过程中的钩子函数，是LangChain1.x 的“王牌”能力
## 真实场景：
想根据问题复杂度动态 切换模型 ；

想 限制 某些用户只能调用部分工具；

想在工具报错时 自动重试 或返回兜底结果；

想在模型调用前 插入额外的系统提示 ；

想记录每一步的 执行日志 ，方便排查问题；

想在敏感信息出现时 阻断执行 ；

想在正式执行工具前增加 人工审批 。

工具链接：https://docs.langchain.com/oss/python/langchain/middleware/overview

# 1 其它12种中间件使用，LangChain提供了16个预置中间件，开箱即用

## 1.1 ModelCallLimitMiddleware中间件
限制模型调用次数，避免无限循环、控制利用成本

## 举例1：测试trigger、keep参数
trigger：是一个列表，每个参数对应一个条件，当满足任意一个条件时除法摘要（token：token数量；message：历史消息数量；fraction：上下文比例，历史消息数量达到模型的max_input_tokens*fraction触发摘要

keep：摘要保存时的原始消息，支持3种（token：摘要时保存token数量；messags：摘要时保存历史消息；fraction：摘要时保留max_input_token*fraction个token）

In [8]:
from typing import List
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage, ToolMessage

from langchain.agents.middleware import SummarizationMiddleware, ModelCallLimitMiddleware, ToolCallLimitMiddleware, \
    ModelFallbackMiddleware, LLMToolSelectorMiddleware, ToolRetryMiddleware
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
import os

from dotenv import load_dotenv
from rich import print as rprint

# 加载配置文件，存在相同key采用当前覆盖
load_dotenv(override=True)

# 具体模型的key和url
DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_API_BASE   = os.getenv("DEEPSEEK_BASE_URL")
DEEPSEEK_MODEL_NAME   = os.getenv("DEEPSEEK_MODEL")


model = init_chat_model(
    model=DEEPSEEK_MODEL_NAME,
    model_provider="deepseek",
    api_key = DEEPSEEK_API_KEY,
    base_url = DEEPSEEK_API_BASE,
    extra_body={"thinking":{"type":"disabled"}}

)


# 创建agent
agent = create_agent(
    model=model,
    checkpointer=InMemorySaver(),
    middleware=[
        ModelCallLimitMiddleware(
            thread_limit=2, #每个线程最多2次调用
            # run_limit=5, #每次运行次数最多5
            # exit_behavior="end", #达到限制后退出
            exit_behavior="error" #达到限制后抛出异常
        )
    ]
)


def pretty_iterate_msg(messages: List[SystemMessage | HumanMessage |
AIMessage | ToolMessage]):
    for msg in messages:
        msg.pretty_print()

config = {"configurable": {"thread_id": "1"}}
response_first = agent.invoke({
"messages": [HumanMessage("你好")]},
config=config
)
print("=" * 30, "> first <", "=" * 30)
pretty_iterate_msg(response_first["messages"])
response_second = agent.invoke({
"messages": [HumanMessage("你是谁？")]},
config=config
)
print("=" * 30, "> second <", "=" * 30)
pretty_iterate_msg(response_second["messages"])
response_third = agent.invoke({
"messages": [HumanMessage("你能帮我做什么？")]},
config=config
)
print("=" * 30, "> third <", "=" * 30)
pretty_iterate_msg(response_third["messages"])



============================== > first < ==============================
================================ Human Message =================================

你好
================================== Ai Message ==================================

你好呀！👋 很高兴见到你！我是DeepSeek，可以帮你解答问题、处理文本、分析文件，或者只是陪你聊聊天。有什么我可以帮你的吗？无论大事小事，尽管说～😊
============================== > second < ==============================
================================ Human Message =================================

你好
================================== Ai Message ==================================

你好呀！👋 很高兴见到你！我是DeepSeek，可以帮你解答问题、处理文本、分析文件，或者只是陪你聊聊天。有什么我可以帮你的吗？无论大事小事，尽管说～😊
================================ Human Message =================================

你是谁？
================================== Ai Message ==================================

嗨！我是**DeepSeek**，由深度求索公司创造的AI助手。😊

简单介绍一下我自己：
- **身份**：纯文本AI模型，目前是DeepSeek最新版本
- **能力**：可以回答问题、处理文档、分析文本、编程辅助、翻译等等
- **特长**：支持上传图像、txt、pdf、ppt、word、excel文件，并从中读取文字信息处理
- **特色**：上下文窗口达1M，可以一次性处理像《三体》

ModelCallLimitExceededError: Model call limits exceeded: thread limit (2/2)

## 1.2 ToolCallLimitMiddleware
限制工具调用次数，可以限制工具总调用次数，也可以限制某个工具调用次数

作用：
避免过多调用某些昂贵的外部API

限制网络爬虫或数据查询请求的数量

避免Agent陷入无限循环

退出有3种模式：
error：直接抛出错误
end：结束整个回话
continue：继续运行Agent，这是默认行为

In [32]:
from langchain.agents import create_agent
from langchain.agents.middleware import ToolCallLimitMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain.messages import SystemMessage, HumanMessage, AIMessage,ToolMessage
from langchain_deepseek import ChatDeepSeek
from pydantic import BaseModel, Field, SecretStr
from typing import List, Union


model = ChatDeepSeek(
    model="any",
    api_base="http://localhost:8889",
    api_key=SecretStr("<KEY>")
)


class ContactInfo(BaseModel):
    """用户的联系方式"""
    name: str = Field(description="用户姓名")
    email: str = Field(description="用户邮箱地址")
    phone: str = Field(description="用户的手机号")
class EventInfo(BaseModel):
    event_name: str = Field(description="事件名称")
    date: str = Field(description="事件发生日期")



    # 创建agent
agent = create_agent(
    model=model,
    checkpointer=InMemorySaver(),
    middleware=[
        ToolCallLimitMiddleware(
            # thread_limit=2, #每个线程最多2次调用
            run_limit=2, #每次运行次数最多2
            # exit_behavior="end", #达到限制后退出
            exit_behavior="error", #达到限制后抛出异常
        )
    ],
    response_format=Union[ContactInfo, EventInfo]
)


def pretty_iterate_msg(messages: List[SystemMessage | HumanMessage |
AIMessage | ToolMessage]):
    for msg in messages:
        msg.pretty_print()
config = {"configurable": {"thread_id": "1"}}
response = agent.invoke({
    "messages": [HumanMessage("你好")]},
    config=config
)
pretty_iterate_msg(response["messages"])

================================ Human Message =================================

你好
================================== Ai Message ==================================
Tool Calls:
  ContactInfo (call_1)
 Call ID: call_1
  Args:
    name: 康师傅
    email: songhongkang@atguigu.cn
    phone: 12345678912
================================= Tool Message =================================
Name: ContactInfo

Returning structured response: name='康师傅' email='songhongkang@atguigu.cn' phone='12345678912'


## 1.2 ModelFallbackMiddleware
用于故障转移，当主模型无法访问时，启动备用模型

### 举例1 调用前中断

In [8]:
from langchain_core.messages import HumanMessage


from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents import create_agent
import os
from langchain.agents.middleware import ModelFallbackMiddleware

from dotenv import load_dotenv

# 加载配置文件，存在相同key采用当前覆盖
load_dotenv(override=True)

# 具体模型的key和url
DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_API_BASE   = os.getenv("DEEPSEEK_BASE_URL")
DEEPSEEK_MODEL_NAME   = os.getenv("DEEPSEEK_MODEL")



# 创建agent
agent = create_agent(
    model="deepseek:faxk",
    tools=[],
    middleware=[
        ModelFallbackMiddleware(
            DEEPSEEK_MODEL_NAME
        )
    ]
)

response = agent.invoke({
    "messages": [HumanMessage("你是谁？")]
})
last_msg = response["messages"][-1]
print(last_msg)
print('=' * 30, '-> model_name <-', '=' * 30)
print(last_msg.response_metadata.get("model_name"))



content='我是 DeepSeek，一个由深度求索公司创造的 AI 助手。我可以帮你解答问题、提供建议、处理文字信息等等。有什么我可以帮你的吗？😊' additional_kwargs={'refusal': None, 'reasoning_content': 'We need answer in Chinese. Need identity. We are DeepSeek Chat V3? Actually user asks "你是谁？" Need respond as AI assistant. Per guidelines no hidden reasoning. Simple.'} response_metadata={'token_usage': {'completion_tokens': 77, 'prompt_tokens': 85, 'total_tokens': 162, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 39, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 85}, 'model_provider': 'deepseek', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'a26a7955944dc5c60445bff77fac9c8e', 'id': '092a7a3d-db76-40af-80d8-a362bb5305e9', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--01a00ea6-974e-7063-b9c0-4a1a2d458bdc-0' tool_calls=[] invali

## 1.2 LLMToolselectorMiddleware
智能工具筛选，当工具太多时，用子模型筛选出最相关的几个模型

参数：

model：子模型

max_tools:限定可以调用工具总数

always_include:指定的工具不被计数

In [12]:
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage


from langchain.agents import create_agent
import os
from langchain.agents.middleware import LLMToolSelectorMiddleware

from dotenv import load_dotenv

# 加载配置文件，存在相同key采用当前覆盖
load_dotenv(override=True)

# 具体模型的key和url
DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_API_BASE   = os.getenv("DEEPSEEK_BASE_URL")
DEEPSEEK_MODEL_NAME   = os.getenv("DEEPSEEK_MODEL")


@tool
def get_weather(city: str):
    """查询指定城市天气"""
    return f"{city}今天天气晴朗"

@tool
def get_news():
    """查询今日国内新闻概要"""
    return ("今日国内新闻概要："
    "中方三艘油轮过航霍尔木兹海峡")
@tool
def calculate(num1: int, num2: int) -> int:
    """
    执行数学计算
    Args:
    num1: 第一个加数
    num2: 第二个加数
    """
    return num1 + num2
@tool
def search_stock(symbol: str):
    """
    查询股票行情
    Args:
    symbol: 股票代码
    """
    return "该股票今天行情不错"


# 通义大模型
# 千问
API_KEY = os.getenv("DASHSCOPE_API_KEY")
BASE_URL   = os.getenv("DASHSCOPE_BASE_URL")
MODEL   = os.getenv("MODEL_NAME")

# 使用了fraction需要设置一个模型上下文大小，DeepSeek的profile为空，需要手动设置
custom_profile = {
    "max_input_tokens": 128_000
}


tongyi_model = init_chat_model(
    model="qwen-plus",
    profile=custom_profile,
    api_key = API_KEY,
    base_url = BASE_URL,
    model_provider="openai",

)

# 创建agent
agent = create_agent(
    model=DEEPSEEK_MODEL_NAME,
    tools=[get_weather, get_news, calculate, search_stock],
    middleware=[
        LLMToolSelectorMiddleware(
            model=tongyi_model,
            max_tools=0,
            # always_include=["get_weather"],
            always_include=["get_news"],
        )
    ]
)

response = agent.invoke({
"messages": HumanMessage("北京今天天气如何？今日新闻概要")
})
for msg in response["messages"]:
    msg.pretty_print()



================================ Human Message =================================

北京今天天气如何？今日新闻概要
================================== Ai Message ==================================
Tool Calls:
  get_news (call_00_UUkVVXcUAAXsZ22Q6FP43682)
 Call ID: call_00_UUkVVXcUAAXsZ22Q6FP43682
  Args:
================================= Tool Message =================================
Name: get_news

今日国内新闻概要：中方三艘油轮过航霍尔木兹海峡
================================== Ai Message ==================================

您好！我来回答您的问题：

## 🌤️ 北京天气
抱歉，我当前没有查询天气的工具，无法获取北京今日的实时天气信息。建议您通过天气App（如中国天气网、墨迹天气等）查询最新预报。

## 📰 今日新闻概要
根据查询到的今日国内新闻：
> **中方三艘油轮过航霍尔木兹海峡**

如需更详细的新闻内容或其他帮助，欢迎继续告诉我！


## 1.4 ToolRetryMiddleware
基于指数退避算法，设置调用工具失败时的重试机制

指数退避（Exponential Backoff） 的核心思想就是：当某个操作失败（通常是网络请求、API 调
用或数据库连接）时，系统不会立刻重试，也不会每次都等待相同的固定时间，而是让每一次重试
的延迟时间按指数级增长。

为什么不直接重试？

想象一下，某个热门网站的服务器因为瞬间流量太大（比如抢票或秒杀）崩溃了。如果所有失败的
客户端都立刻或每隔1秒就重试一次，这无异于对已经瘫痪的服务器进行了一场持续的 DDoS（分
布式拒绝服务）攻击，服务器可能永远也缓不过来。

### 举例1

In [17]:
import datetime
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage


from langchain.agents import create_agent
import os
from langchain.agents.middleware import ToolRetryMiddleware

from dotenv import load_dotenv

# 加载配置文件，存在相同key采用当前覆盖
load_dotenv(override=True)

# 具体模型的key和url
DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_API_BASE   = os.getenv("DEEPSEEK_BASE_URL")
DEEPSEEK_MODEL_NAME   = os.getenv("DEEPSEEK_MODEL")



def write_times(s):
    """将每次工具调用的时间戳和间隔写入本地文件，方便观察退避策略"""
    with open("call_times_with_jitter.txt", "a", encoding="utf-8") as f:
        f.write(s + "\n")

count = 1
start_time = None

@tool
def get_weather(city: str):
    """查询指定城市天气"""
    global count
    global start_time
    interval = 0
    current_time = datetime.datetime.now()
    if not start_time:
        interval = 0
    else:
    # 计算当前调用与上一次调用之间的时间差（秒）
        interval = (current_time - start_time).total_seconds()
    start_time = current_time
    res_str = f"第 {count} 次调用，当前时间： {start_time}, 和上次调用间隔{interval} 秒"
    count += 1
    # 记录日志
    write_times(res_str)
    # 故意抛出 TimeoutError，以此触发中间件的重试机制
    raise TimeoutError("Not Implemented")


tongyi_model = init_chat_model(
    model="qwen-plus",
    profile=custom_profile,
    api_key = API_KEY,
    base_url = BASE_URL,
    model_provider="openai",

)

# 创建agent
agent = create_agent(
    model=DEEPSEEK_MODEL_NAME,
    tools=[get_weather],
    middleware=[
        ToolRetryMiddleware(
        max_retries=6, # 最大重试次数（不包含初始的那次调用，一共最多调 1 + 6 =7 次）
        backoff_factor=2.0, # 指数退避因子（每次重试等待时间乘以 2）
        initial_delay=1.0, # 第一次重试前的初始等待时间（1 秒）
        max_delay=10.0, # 最大等待延迟上限（防止指数增长无限大，限制在 10 秒）
        jitter=True, # 开启抖动（在等待时间中加入随机性，防止并发请求时出现“惊群效应”）
        retry_on=(TimeoutError,), # 仅针对捕获到特定的 TimeoutError 异常时才触发重试
        on_failure="continue" # 当达到最大重试次数依然失败时，Agent 的行为："continue" 表示将错误信息包装后塞回对话历史，让大模型知道失败了并继续决策
        )
    ]
)

response = agent.invoke({
"messages": [HumanMessage("今天北京天气如何？")]
})
# 1. 你的提问 -> 2. AI 决定调用工具 -> 3. 重试失败后的错误反馈 -> 4. AI 最终给出的兜底回复
for msg in response["messages"]:
    msg.pretty_print()



================================ Human Message =================================

今天北京天气如何？
================================== Ai Message ==================================

我来帮您查询北京的天气情况。
Tool Calls:
  get_weather (call_00_gqYhl9v9Nd9nBG3HqSBO6672)
 Call ID: call_00_gqYhl9v9Nd9nBG3HqSBO6672
  Args:
    city: 北京
================================= Tool Message =================================
Name: get_weather

Tool 'get_weather' failed after 7 attempts with TimeoutError: Not Implemented. Please try again.
================================== Ai Message ==================================

很抱歉，我尝试查询北京的天气，但天气服务暂时出现了故障（请求超时），未能成功获取到天气数据。

建议您可以：
1. **稍后重试** - 可能是服务暂时繁忙，过一会儿再问我试试
2. **通过其他渠道查询** - 比如中国天气网（weather.com.cn）或手机上的天气应用

如果您需要，我可以稍后再次帮您尝试查询。请问还有其他可以帮您的吗？


## 1.5 ModelRetryMiddleware
模型调用失败时，基于指数退避算法，策略和工具一样

In [19]:
import datetime
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage


from langchain.agents import create_agent
import os
from langchain.agents.middleware import ModelRetryMiddleware

from dotenv import load_dotenv

# 加载配置文件，存在相同key采用当前覆盖
load_dotenv(override=True)

# 具体模型的key和url
DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_API_BASE   = os.getenv("DEEPSEEK_BASE_URL")
DEEPSEEK_MODEL_NAME   = os.getenv("DEEPSEEK_MODEL")



def write_times(s):
    """将每次工具调用的时间戳和间隔写入本地文件，方便观察退避策略"""
    with open("call_times_with_jitter.txt", "a", encoding="utf-8") as f:
        f.write(s + "\n")

count = 1
start_time = None

@tool
def get_weather(city: str):
    """查询指定城市天气"""
    global count
    global start_time
    interval = 0
    current_time = datetime.datetime.now()
    if not start_time:
        interval = 0
    else:
    # 计算当前调用与上一次调用之间的时间差（秒）
        interval = (current_time - start_time).total_seconds()
    start_time = current_time
    res_str = f"第 {count} 次调用，当前时间： {start_time}, 和上次调用间隔{interval} 秒"
    count += 1
    # 记录日志
    write_times(res_str)
    # 故意抛出 TimeoutError，以此触发中间件的重试机制
    raise TimeoutError("Not Implemented")


tongyi_model = init_chat_model(
    model="qwen-plus",
    profile=custom_profile,
    api_key = API_KEY,
    base_url = BASE_URL,
    model_provider="openai",

)

# 创建agent
agent = create_agent(
    model="deepseek:xs",
    tools=[get_weather],
    middleware=[
        ModelRetryMiddleware(
        max_retries=6, # 最大重试次数（不包含初始的那次调用，一共最多调 1 + 6 =7 次）
        backoff_factor=2.0, # 指数退避因子（每次重试等待时间乘以 2）
        initial_delay=1.0, # 第一次重试前的初始等待时间（1 秒）
        max_delay=10.0, # 最大等待延迟上限（防止指数增长无限大，限制在 10 秒）
        jitter=True, # 开启抖动（在等待时间中加入随机性，防止并发请求时出现“惊群效应”）
        retry_on=(TimeoutError,), # 仅针对捕获到特定的 TimeoutError 异常时才触发重试
        on_failure="error" # 当达到最大重试次数依然失败时，Agent 的行为："continue" 表示将错误信息包装后塞回对话历史，让大模型知道失败了并继续决策
        )
    ]
)

response = agent.invoke({
"messages": [HumanMessage("你好？")]
})
# 1. 你的提问 -> 2. AI 决定调用工具 -> 3. 重试失败后的错误反馈 -> 4. AI 最终给出的兜底回复
for msg in response["messages"]:
    msg.pretty_print()



BadRequestError: Error code: 400 - {'error': {'message': 'The supported API model names are deepseek-v4-pro or deepseek-v4-flash, but you passed xs.', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_request_error'}}

##  1.6 LLMToolEmulator
在某些情况下，工具尚未开发完成，希望进行先测试，可以用词工具

In [20]:
from langchain.agents import create_agent
from langchain.agents.middleware import LLMToolEmulator
from langchain.messages import HumanMessage
@tool
def get_weather(city: str):
    """查询指定城市天气"""
    return f"{city}今天天气晴朗"
agent = create_agent(
    model=model,
    tools=[get_weather],
    middleware=[
    LLMToolEmulator(
        model=tongyi_model,
    )
    ]
)
response = agent.invoke({
"messages": [HumanMessage("今天北京天气如何")]
})
for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

今天北京天气如何
================================== Ai Message ==================================

我来帮你查询北京的天气情况。
Tool Calls:
  get_weather (call_00_sAVGsy5LPV5w0g4fTZ9T8223)
 Call ID: call_00_sAVGsy5LPV5w0g4fTZ9T8223
  Args:
    city: 北京
================================= Tool Message =================================
Name: get_weather

北京今日天气：晴，气温12°C～24°C，西北风3级，空气质量良，紫外线中等。建议午后适当补水，早晚温差较大，请注意增减衣物。
================================== Ai Message ==================================

北京今天的天气情况如下：

**☀️ 今日天气：晴**
- **气温**：12°C ～ 24°C
- **风力**：西北风3级
- **空气质量**：良
- **紫外线**：中等

**温馨提示：**
- 早晚温差较大（12°C至24°C），建议根据时段适当增减衣物
- 午后记得适当补水
- 紫外线中等，外出可做好基本防晒

祝你有愉快的一天！😊


## 1.7 ContextEditingMiddleware
上下文编辑中间件，该中间件提供了上下文管理的一种方式。
通过更改发送给模型的消息列表来控制成本。
1. ContextEditingMiddleware 的价值：大模型多轮对话时，如果频繁调用产生大量文本的工具
（如代码执行、网页爬取），历史记录会急剧膨胀。这个中间件就像一个“上下文抽脂手术”，在不
影响当前对话的前提下，自动在后台删掉之前沉淀的工具调用废话，从而极大地节省 Token 费用
并防止超出模型最大上下文窗口（Context Window）。
2. InMemorySaver ：它在内存中开辟了一个空间。第二轮和第三轮提问时，Agent 能通过
thread_id 自动找回前几轮的记忆。

In [23]:
from langchain.agents.middleware import ContextEditingMiddleware, ClearToolUsesEdit

count = 0

@tool
def get_weather(city: str):
    """查询指定城市天气"""
    global count
    # 故意返回一段非常冗长、包含大量 Token 的文本，用于测试中间件的 Token 清理/截断功能
    return (f"当前是第 {count} 次调用工具，{city}今天天气晴朗"
    f"天气非常好，北风，非常适合出行，盼望着，盼望着，"
    f"春天来了。我喜欢春天，你喜欢吗，天气真的很不错"
    f"万里无云，天气晴朗，春和景明，哈哈哈哈哈哈，这是凑字数的"
            f"真不错，天气非常好，适合出行，这里token挺多的"
    f"可以出门玩，尅有跑步，钓鱼，爬山，一切都很好哈哈哈")

agent = create_agent(
    model=model,
    tools=[get_weather],
    middleware=[
    ContextEditingMiddleware(
        edits=[
        ClearToolUsesEdit(
        trigger=50,
        keep=0,
        ),
        ],
    ),
    ],
    checkpointer=InMemorySaver()
)
config = {"configurable": {"thread_id": "1"}}
for i in range(3):
    print("=" * 30, f"当前是第 {i + 1} 轮调用", "=" * 30)
    count = i + 1
    response = agent.invoke({
        "messages": [HumanMessage(f"第 {i + 1} 次询问：今天北京天气如何，一句话回答")]},
        config=config
    )
    print("---- 本次返回的 messages ----")
    for msg in response["messages"]:
        # msg.pretty_print()
        if isinstance(msg, AIMessage):
            if not msg.tool_calls:
                print(f"本次token用量：{msg.usage_metadata}")

============================== 当前是第 1 轮调用 ==============================
---- 本次返回的 messages ----
本次token用量：{'input_tokens': 345, 'output_tokens': 15, 'total_tokens': 360, 'input_token_details': {'cache_read': 256}, 'output_token_details': {}}
============================== 当前是第 2 轮调用 ==============================
---- 本次返回的 messages ----
本次token用量：{'input_tokens': 345, 'output_tokens': 15, 'total_tokens': 360, 'input_token_details': {'cache_read': 256}, 'output_token_details': {}}
本次token用量：{'input_tokens': 437, 'output_tokens': 20, 'total_tokens': 457, 'input_token_details': {'cache_read': 384}, 'output_token_details': {}}
============================== 当前是第 3 轮调用 ==============================
---- 本次返回的 messages ----
本次token用量：{'input_tokens': 345, 'output_tokens': 15, 'total_tokens': 360, 'input_token_details': {'cache_read': 256}, 'output_token_details': {}}
本次token用量：{'input_tokens': 437, 'output_tokens': 20, 'total_tokens': 457, 'input_token_details': {'cache_read': 384}, 'out

对照组-不裁剪上下文

In [24]:
from langchain.agents.middleware import ContextEditingMiddleware, ClearToolUsesEdit

count = 0

@tool
def get_weather(city: str):
    """查询指定城市天气"""
    global count
    # 故意返回一段非常冗长、包含大量 Token 的文本，用于测试中间件的 Token 清理/截断功能
    return (f"当前是第 {count} 次调用工具，{city}今天天气晴朗"
    f"天气非常好，北风，非常适合出行，盼望着，盼望着，"
    f"春天来了。我喜欢春天，你喜欢吗，天气真的很不错"
    f"万里无云，天气晴朗，春和景明，哈哈哈哈哈哈，这是凑字数的"
            f"真不错，天气非常好，适合出行，这里token挺多的"
    f"可以出门玩，尅有跑步，钓鱼，爬山，一切都很好哈哈哈")

agent = create_agent(
    model=model,
    tools=[get_weather],
    checkpointer=InMemorySaver()
)
config = {"configurable": {"thread_id": "1"}}
for i in range(3):
    print("=" * 30, f"当前是第 {i + 1} 轮调用", "=" * 30)
    count = i + 1
    response = agent.invoke({
        "messages": [HumanMessage(f"第 {i + 1} 次询问：今天北京天气如何，一句话回答")]},
        config=config
    )
    print("---- 本次返回的 messages ----")
    for msg in response["messages"]:
        # msg.pretty_print()
        if isinstance(msg, AIMessage):
            if not msg.tool_calls:
                print(f"本次token用量：{msg.usage_metadata}")

============================== 当前是第 1 轮调用 ==============================
---- 本次返回的 messages ----
本次token用量：{'input_tokens': 438, 'output_tokens': 15, 'total_tokens': 453, 'input_token_details': {'cache_read': 256}, 'output_token_details': {}}
============================== 当前是第 2 轮调用 ==============================
---- 本次返回的 messages ----
本次token用量：{'input_tokens': 438, 'output_tokens': 15, 'total_tokens': 453, 'input_token_details': {'cache_read': 256}, 'output_token_details': {}}
本次token用量：{'input_tokens': 616, 'output_tokens': 15, 'total_tokens': 631, 'input_token_details': {'cache_read': 512}, 'output_token_details': {}}
============================== 当前是第 3 轮调用 ==============================
---- 本次返回的 messages ----
本次token用量：{'input_tokens': 438, 'output_tokens': 15, 'total_tokens': 453, 'input_token_details': {'cache_read': 256}, 'output_token_details': {}}
本次token用量：{'input_tokens': 616, 'output_tokens': 15, 'total_tokens': 631, 'input_token_details': {'cache_read': 512}, 'out

## 1.9 FilesystemFileSearchMiddleware中间件

基于系统的Glob和Grep检索工具，为Agent赋予本地文件搜索和分析的能力。

Glob根据文件路径检索

Grep根据文件内容检索

In [27]:
from langchain.agents.middleware import FilesystemFileSearchMiddleware



agent = create_agent(
    model=model,
    tools=[],
    middleware=[
        FilesystemFileSearchMiddleware(
        root_path="../todo_workspace", #搜索目录
        # 【可选】限制搜索的文件后缀，防止模型读取非代码或无关文件
        # allowed_extensions=[".py", ".ipynb", ".js", ".md"], # 允许的文件类型
        # 是否启用 ripgrep 搜索引擎：
        # 设为 True 可以获得比原生 Grep 更快的性能（前提是系统已安装 ripgrep）
        use_ripgrep=True,
        # 单个文件的最大读取限制（单位MB）：防止读取超大型日志或二进制文件导致 OOM
        max_file_size_mb=10
        ),
    ],
)
result = agent.invoke({
    "messages": [HumanMessage("找到包含add函数的Python或Jupyter文件")]
})
for msg in result["messages"]:
    msg.pretty_print()

================================ Human Message =================================

找到包含add函数的Python或Jupyter文件
================================== Ai Message ==================================

我来帮你找到包含 `add` 函数的 Python 或 Jupyter 文件。
Tool Calls:
  grep_search (call_00_s33C2TzyBLEwKgk9Xfcy3021)
 Call ID: call_00_s33C2TzyBLEwKgk9Xfcy3021
  Args:
    pattern: def add\s*\(
    include: *.py
    output_mode: files_with_matches
  grep_search (call_01_ObWpF2qq3YvdADfzsiAC2692)
 Call ID: call_01_ObWpF2qq3YvdADfzsiAC2692
  Args:
    pattern: def add\s*\(
    include: *.ipynb
    output_mode: files_with_matches
================================= Tool Message =================================
Name: grep_search

No matches found
================================= Tool Message =================================
Name: grep_search

No matches found
================================== Ai Message ==================================

我搜索了包含 `def add(` 定义的 Python (`.py`) 和 Jupyter (`.ipynb`) 文件，但没有找到匹配的文件。

让我换一

## 1.8 Shell tool中间件
为Agent提供一个可以执行命令的Shell环境。
Windows下无法测试

## 1.9 Filesystem中间件
这是源自deepagents（基于LangChain的另一个框架）的中间件
内置了四个工具，分别用于查看目录、读文件、写文件和改文件

## 1.9 Subagent中间件
也是来自deepagents的中间件
用于便捷地创建子Agent

# 2.0 自定义中间件
某些复杂场景下，官方内置的中间件不能完全满足需求，此时可以通过实现LangChain暴露的中间件
hook函数 构建自定义中间件。

Hook 函数，中文常叫 钩子函数 ，指的是：在某个既定流程的特定时机，被框架、系统或主程序 自动
调用 的扩展函数。

核心特点：
1、不是你主动在业务代码里随便调用的，而是当流程运行到某个“钩子点”时，系统自动触发它。

2、它依附于一个更大的执行流程。比如“请求开始前”“模型调用前”“任务结束后”“异常发生时”等。

3、它的作用是让你在不改主流程源码的前提下插入自己的逻辑。例如做 日志 、 鉴权 、 修改输入 、
拦截输出 、 清理资源 等。

LangChain的中间件作用在Agent架构中，后者是基于LangGraph构建的流程图。如下列出了六个hook
函数（钩子函数）：

## 2.1 LangChain的hook函数分类
官方将六个钩子函数按照风格分为两类

类型1：Node-style hooks(节点风格钩子)
顾名思义，它们在流程的 特定节点 运行。
适合顺序逻辑，如记录日志、验证
包括

before_agent：在Agent开始运行之前执行。

before_model：在模型调用之前执行。

after_model：在模型调用之后执行。

after_agent：在Agent流程全部完成后执行。

类型2：Wrap-style hooks(包装风格钩子)
顾名思义，它们在 模型或工具调用前后 运行。
适合控制流，如重试、回退、缓存。
包括
wrap_model_call (包裹模型调用)

wrap_tool_call (包裹工具调用)


## 2.2 Node-style hooks 函数用法
支持两种用法：
装饰器的函数挂载，把一个hook快速挂载到agent的某个节点

类写法对象中间件：把中间件封装为一个可配置，可扩展的组件

### 2.2.1 基本用法
1.0基于装饰器

In [28]:
from langchain.agents.middleware import before_model, after_model,before_agent, after_agent, AgentState, AgentMiddleware
from langchain.messages import HumanMessage
from langgraph.runtime import Runtime
from langchain.agents import create_agent
from typing import Any
# 1. 定义 before_model 钩子
@before_model
def before_model_middleware(state: AgentState, runtime: Runtime) ->dict[str, Any] | None:
    state["messages"][-1].content += " -> before_model <- "
    return None
# 2. 定义 after_model 钩子
@after_model
def after_model_middleware(state: AgentState, runtime: Runtime) -> dict[str,
    Any] | None:
    state["messages"][-1].content += " -> after_model <- "
    return None
# 3. 定义 before_agent 钩子
@before_agent
def before_agent_middleware(state: AgentState, runtime: Runtime) ->dict[str, Any] | None:
    state["messages"][-1].content += " -> before_agent <- "
    return None
# 4. 定义 after_agent 钩子
@after_agent
def after_agent_middleware(state: AgentState, runtime: Runtime) -> None:
    state["messages"][-1].content += " -> after_agent <- "
    return None

agent = create_agent(
model = model,
middleware = [before_model_middleware, after_model_middleware,
before_agent_middleware, after_agent_middleware] # 👈 添加中间件
)
response = agent.invoke({
"messages": [HumanMessage("你好啊")],
})
for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

你好啊 -> before_agent <-  -> before_model <- 
================================== Ai Message ==================================

你好呀！😊 有什么我可以帮你的吗？无论是聊天、解答问题，还是需要一些建议，我都很乐意为你提供帮助！随时告诉我你的需求哦~ -> after_model <-  -> after_agent <-


基于类实现
关键规则：
1. 必须继承 AgentMiddleware ← 这个固定
2. 方法名固定 ( before_model , after_model ) ← 这个固定
3. 类名随意 ← 这个不固定

LangGraph 只看：

是否继承 AgentMiddleware？

是否有 before_model / after_model 等方法？

In [29]:
class MyMiddleware(AgentMiddleware):
    def __init__(self):
        super().__init__()
    def before_model(self, state: AgentState, runtime: Runtime) -> dict[str,Any] | None:
        state["messages"][-1].content += " -> before_model <- "
        return None
    def after_model(self, state: AgentState, runtime: Runtime) -> dict[str,
        Any] | None:
        state["messages"][-1].content += " -> after_model <- "
        return None
    def before_agent(self, state: AgentState, runtime: Runtime) -> dict[str,Any] | None:
        state["messages"][-1].content += " -> before_agent <- "
        return None
    def after_agent(self, state: AgentState, runtime: Runtime) -> None:
        state["messages"][-1].content += " -> after_agent <- "
        return None

my_middleware = MyMiddleware()
agent = create_agent(
model = model,
middleware = [my_middleware]
)


response = agent.invoke({
"messages": [HumanMessage("你好啊")],
})
for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

你好啊 -> before_agent <-  -> before_model <- 
================================== Ai Message ==================================

你好啊！👋 很高兴见到你～

我是DeepSeek，随时准备帮你解答问题、处理任务或者聊聊天。不管是学习、工作还是生活中的困惑，或者只是想找人说说话，我都在这儿呢！

今天有什么我可以帮你的吗？或者有什么想聊的话题？😊 -> after_model <-  -> after_agent <-


1. before_model 通常的场景：
    消息修剪（trim messages）
    PII 脱敏
    输入验证
    条件路由

2. after_model 通常的场景：
    输出验证
    格式化响应
    统计信息
    状态更新

## 2.3 Wrap-style hooks函数用法

### 2.3.1 基本用法
基于装饰器实现 可以同时在模型调用前后做事，所以命名为 wrap_model_call ，wrap意为 包裹 。

In [32]:
from langchain.agents.middleware import wrap_model_call, ModelRequest,ModelResponse
from langchain.messages import HumanMessage
from langchain.agents import create_agent
from typing import Callable
@wrap_model_call
def wrap_model_call_middleware(request: ModelRequest, # 包含即将发送给大模型的所有请求数据（如消息列表、温度等）
                               handler: Callable[[ModelRequest], ModelResponse], # 核心句柄：代表下一个中间件或最终的大模型调用服务
    ) -> ModelResponse | None:
        # 动态篡改用户发出的最后一条消息的内容，悄悄往里面追加字符串。
        # 典型应用：统一在底层为所有请求追加特殊的 Prompt 提示词（例如：“请用中文回答”、“禁止透漏公司机密”等）。
        request.messages[-1].content += " -> wrap_model_call_before <- "
        # 将修改后的请求传递给 handler，真正去调用大模型（或者流转到下一个拦截器）
        # 这一步会产生真实的 Token 消耗并等待大模型响应
        response = handler(request)
        # 大模型返回响应后，在将响应交付给 Agent 状态机之前，对其内容进行直接篡改
        # `response.result` 是一个消息列表，修改其第一条返回消息的内容
        # 典型应用：做底层的文本敏感词过滤、输出格式强行格式化、或是统一添加某些后处理标记。
        response.result[0].content += " -> wrap_model_call_after <- "
        # 将修改完的响应体返回，继续维持 Agent 生命周期流转
        return response

agent = create_agent(
    model = model,
    middleware = [wrap_model_call_middleware]
)
response = agent.invoke({
"messages": [HumanMessage("你好啊")],
})
for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

你好啊 -> wrap_model_call_before <- 
================================== Ai Message ==================================

你好呀！👋 很高兴见到你！

我是DeepSeek，你的AI助手。无论你想聊聊天、问问题、寻求建议，还是需要帮忙解决某个具体问题，我都会尽力帮助你。

有什么我可以为你做的吗？不用客气，尽管说！😊 -> wrap_model_call_after <-


基于类实现

In [33]:
from langchain.agents.middleware import wrap_model_call, ModelRequest,ModelResponse
from langchain.messages import HumanMessage
from langchain.agents import create_agent
from typing import Callable

class wrap_model_call_middleware(AgentMiddleware):
    def wrap_model_call(
        self,
        request: ModelRequest,
        handler: Callable[[ModelRequest], ModelResponse],
        ) -> ModelResponse | None:
        request.messages[-1].content += " -> wrap_model_call_before <- "
        response = handler(request)
        response.result[0].content += " -> wrap_model_call_after <- "
        return response


agent = create_agent(
    model = model,
    middleware = [wrap_model_call_middleware()]
)
response = agent.invoke({
"messages": [HumanMessage("你好啊")],
})
for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

你好啊 -> wrap_model_call_before <- 
================================== Ai Message ==================================

你好！很高兴见到你！😊

有什么我可以帮你的吗？无论是想聊聊天、解答疑问，还是需要一些建议，我都在这儿呢。尽管说吧！ -> wrap_model_call_after <-


使用场景：用于拦截、重试、缓存模型调用

重试场景

In [35]:
from langchain.agents.middleware import wrap_model_call, ModelRequest,ModelResponse
from typing import Callable
import time
@wrap_model_call
def retry_model(
    request: ModelRequest,
    handler: Callable[[ModelRequest], ModelResponse]
    ) -> ModelResponse:
        """自动重试失败的模型调用"""
        max_retries = 3
        for attempt in range(max_retries):
            try:
                print(f"🔄 尝试调用模型（第 {attempt + 1}/{max_retries} 次）")
                return handler(request)
            except Exception as e:
                if attempt == max_retries - 1:
                    print(f"❌ 所有重试失败：{e}")
                    raise
                    # 指数退避
                    wait_time = 2 ** attempt
                    print(f"⚠ 调用失败：{e}，{wait_time} 秒后重试")
                    time.sleep(wait_time)

响应缓存

In [37]:
from langchain.agents.middleware import wrap_model_call, ModelRequest,ModelResponse
from typing import Callable
import hashlib
import json

class ModelCache:
    """模型响应缓存"""
    def __init__(self):
        self.cache = {}
    def create_hook(self):

        @wrap_model_call
        def cache_model(
            request: ModelRequest,
            handler: Callable[[ModelRequest], ModelResponse]
            ) -> ModelResponse:
            # 生成缓存键
            cache_key = hashlib.md5(
                json.dumps({
                "messages": [str(m) for m in request.messages],
                "system": str(request.system_message)
                }).encode()
            ).hexdigest()
            # 检查缓存
            if cache_key in self.cache:
                print("💾 缓存命中！")
                return self.cache[cache_key]
            # 调用模型
            print("🔍 缓存未命中，调用模型")
            response = handler(request)
            # 存入缓存
            self.cache[cache_key] = response
            return response
        return cache_model

    # 使用
cache = ModelCache()
agent = create_agent(
model=model,
middleware=[cache.create_hook()]
)

修改系统提示词

In [38]:
from langchain.agents.middleware import wrap_model_call, ModelRequest,ModelResponse
from langchain_core.messages import SystemMessage
from typing import Callable
@wrap_model_call
def add_context(
    request: ModelRequest,
    handler: Callable[[ModelRequest], ModelResponse]
    ) -> ModelResponse:
    """动态添加上下文信息到系统提示"""
    # 获取当前时间
    from datetime import datetime
    current_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    # 构建新的系统消息
    original_content = request.system_message.content if request.system_message else ""
    new_content = f"""{original_content}
    当前时间：{current_time}
    用户位置：中国
    语言偏好：中文
    """
    # 创建新的系统消息
    new_system_message = SystemMessage(content=new_content)
    # 使用 override 方法修改请求
    modified_request = request.override(system_message=new_system_message)
    return handler(modified_request)

wrap_tool_call
可以同时在工具调用前后做事，所以命名为 wrap_tool_call 。

基于装饰器实现

In [39]:
from langchain.agents.middleware import wrap_tool_call
from langchain.tools.tool_node import ToolCallRequest
from langchain.messages import HumanMessage, ToolMessage
from langchain.agents import create_agent
from langchain.tools import tool
from langgraph.types import Command
from typing import Callable
@tool
def get_weather(city: str, is_forcast: bool) -> str:
    """
    获取当日特定城市的天气

    Args:
    city: 城市名称
    is_forcast: 是否包含明天的天气预报
    """
    res = f"{city}今天天气不错"
    if is_forcast:
        res += "\n明天天气也很好"
    return res
@wrap_tool_call
def wrap_tool_call_middleware(
    request: ToolCallRequest,
    handler: Callable[[ToolCallRequest], ToolMessage | Command],
    ) -> ToolMessage | Command:
    result = handler(request)
    print(f"原始参数：{request.tool_call['args']}")
    print(f"原始参数调用结果： {result}")
    request.tool_call["args"]["is_forcast"] = True
    result = handler(request)
    print(f"更新后的参数：{request.tool_call['args']}")
    print(f"更新参数调用结果： {result}")
    return result
agent = create_agent(
    model=model,
    tools=[get_weather],
    middleware=[wrap_tool_call_middleware]
)
response = agent.invoke({
"messages": [HumanMessage("你好啊，今天杭州的天气怎么样")],
    })
for msg in response["messages"]:
    msg.pretty_print()

原始参数：{'city': '杭州', 'is_forcast': False}
原始参数调用结果： content='杭州今天天气不错' name='get_weather' tool_call_id='call_00_vLE6gFzv8Juusi0gh1Tt0612'
更新后的参数：{'city': '杭州', 'is_forcast': True}
更新参数调用结果： content='杭州今天天气不错\n明天天气也很好' name='get_weather' tool_call_id='call_00_vLE6gFzv8Juusi0gh1Tt0612'
================================ Human Message =================================

你好啊，今天杭州的天气怎么样
================================== Ai Message ==================================

我来帮您查询一下杭州今天的天气情况。
Tool Calls:
  get_weather (call_00_vLE6gFzv8Juusi0gh1Tt0612)
 Call ID: call_00_vLE6gFzv8Juusi0gh1Tt0612
  Args:
    city: 杭州
    is_forcast: True
================================= Tool Message =================================
Name: get_weather

杭州今天天气不错
明天天气也很好
================================== Ai Message ==================================

查询到啦！😊

**杭州今日天气**：天气不错，是个适合出门的好日子！

另外我还顺便看了明天的天气，**明天天气也很好**，如果您有出行计划的话，这两天都是不错的选择呢。

祝您在杭州有个愉快的一天！如需了解更多信息，随时可以问我～


基于类实现

In [41]:
from langchain.agents.middleware import AgentMiddleware
from langchain.tools.tool_node import ToolCallRequest
from langchain.messages import HumanMessage, ToolMessage
from langchain.agents import create_agent
from langchain.tools import tool
from langgraph.types import Command
from typing import Callable
@tool
def get_weather(city: str, is_forcast: bool) -> str:
    """
    获取当日特定城市的天气
    Args:
    city: 城市名称
    is_forcast: 是否包含明天的天气预报
    """
    res = f"{city}今天天气不错"
    if is_forcast:
        res += "\n明天天气也很好"
    return res

class WrapToolCallMiddleware(AgentMiddleware):
    def wrap_tool_call(
        self,
        request: ToolCallRequest,
        handler: Callable[[ToolCallRequest], ToolMessage | Command],
        ) -> ToolMessage | Command:
        result = handler(request)
        print(f"原始参数：{request.tool_call['args']}")
        print(f"原始参数调用结果： {result}")
        request.tool_call["args"]["is_forcast"] = True
        result = handler(request)
        print(f"更新后的参数：{request.tool_call['args']}")
        print(f"更新参数调用结果： {result}")
        return result

agent = create_agent(
model = model,
tools = [get_weather],
middleware = [WrapToolCallMiddleware()]
)
response = agent.invoke({
    "messages": [HumanMessage("你好啊，今天杭州的天气怎么样")],
})
for msg in response["messages"]:
    msg.pretty_print()

原始参数：{'city': '杭州', 'is_forcast': False}
原始参数调用结果： content='杭州今天天气不错' name='get_weather' tool_call_id='call_00_dpGxnPzu65eOiRhb3hlw2304'
更新后的参数：{'city': '杭州', 'is_forcast': True}
更新参数调用结果： content='杭州今天天气不错\n明天天气也很好' name='get_weather' tool_call_id='call_00_dpGxnPzu65eOiRhb3hlw2304'
================================ Human Message =================================

你好啊，今天杭州的天气怎么样
================================== Ai Message ==================================

好的，我来帮你查询一下今天杭州的天气情况。
Tool Calls:
  get_weather (call_00_dpGxnPzu65eOiRhb3hlw2304)
 Call ID: call_00_dpGxnPzu65eOiRhb3hlw2304
  Args:
    city: 杭州
    is_forcast: True
================================= Tool Message =================================
Name: get_weather

杭州今天天气不错
明天天气也很好
================================== Ai Message ==================================

你好！我帮你查了一下杭州今天的天气：

**今天杭州天气不错** ☀️

另外，我顺便也查了明天的天气，**明天天气也很好**，看来杭州这两天的天气都挺适合出行的！

请问还有什么需要我帮忙的吗？需要我帮你查询其他城市或者了解更多天气细节吗？


## 2.3 装饰器和类的选择器
情况1：中间件只用一个钩子函数，推荐用装饰器，需要多个钩子函数推荐类写法

当一个中间件只需要实现一个钩子函数时，直接使用装饰器最简单。

当一个中间件需要实现多个钩子函数时，类写法更合适。

装饰器也不是不能实现，多数情况下可以像下面的示例里那样通过工厂函数返回多个装饰器函数来完
成；但这种方式本质上是把一个“逻辑上属于同一个中间件”的行为拆成多个独立函数，再由外部统一组
装，因此不如类写法自然、集中、清晰。